# Explore here

In [1]:
!pip install requests pandas matplotlib seaborn sqlalchemy

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [14]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

paises = ['chn', 'ago', 'ken', 'usa','mex']
indicadores = {'SP.POP.TOTL':'Poblacion_Total',
               'NY.GDP.MKTP.KD.ZG': 'GDP_Growth_Anual_%',
               #'NY.GDP.PCAP.CD' : 'PIB_per_capita_(USD_actuales)',
               #'EN.GHG.C02.PC.CE.AR5':'CO2_Per_Capita', 
               'SP.DYN.CBRT.IN': 'Natalidad_por_1000_habitantes'
               }


#for ind in indicadores:
#    url = f'https://api.worldbank.org/v2/country/{";".join(paises)}/indicator/{ind}?date=2010:2024&format=json'
#    response = requests.get(url)
#    print(response.json())


def feth_data_indicador (codigo_pais, id_indicador, fecha_inicio=2010, fecha_fin=2024):

    paises = ";".join(codigo_pais)
    endpoint = f'https://api.worldbank.org/v2/country/{paises}/indicator/{id_indicador}'
    pagina = 1
    datos_extendidos = []

    while True:
        params = {
            'format' : 'json',
            'date' : f'{fecha_inicio}:{fecha_fin}',
            'page' : pagina
        }
        response = requests.get(endpoint, params=params)
        data = response.json()

        if not isinstance(data,list) or len(data)==0:
            raise ValueError(f'Respuesta API no esperada para{id_indicador}:{data}')
    
        metadatos = data[0]
        datos = data[1]
        datos_extendidos.extend(datos)

        total_paginas = metadatos.get('pages')

        if pagina >= total_paginas:
            break
        pagina+=1
    
    return datos_extendidos



In [16]:
import pandas as pd

tablas = {}

for id_indicador, nombre_tabla in indicadores.items():
    data_raw = feth_data_indicador(paises,id_indicador)

    records = []
    for fila in data_raw:
        records.append({
            'pais': fila['country']['value'],
            'agno' : fila['date'],
            'valor': fila['value']
        })
    df = pd.DataFrame(records)
    df['agno'] = pd.to_numeric(df['agno'], errors='coerce').astype('Int64')
    tablas[nombre_tabla] = df


    


#tablas['GDP_Growth_Anual_%'].head()

tablas.items()

dict_items([('Poblacion_Total',              pais  agno      valor
0          Angola  2024   37885849
1          Angola  2023   36749906
2          Angola  2022   35635029
3          Angola  2021   34532429
4          Angola  2020   33451132
..            ...   ...        ...
70  United States  2014  319257560
71  United States  2013  316726282
72  United States  2012  314339099
73  United States  2011  311839461
74  United States  2010  309378227

[75 rows x 3 columns]), ('GDP_Growth_Anual_%',              pais  agno     valor
0          Angola  2024  4.423907
1          Angola  2023  1.263308
2          Angola  2022  4.216003
3          Angola  2021  2.102753
4          Angola  2020 -4.042447
..            ...   ...       ...
70  United States  2014  2.523820
71  United States  2013  2.117830
72  United States  2012  2.289113
73  United States  2011  1.564407
74  United States  2010  2.695193

[75 rows x 3 columns]), ('Natalidad_por_1000_habitantes',              pais  agno   valor
0